In [1]:
# !pip install uncertainties

$$
\epsilon_e = (1/2)\left(\epsilon_r+1 + (\epsilon_r-1)\left[\frac{1}{\sqrt{1+12h_s/w_r}}\right]\right)
$$

In [9]:
# from numpy import sqrt
import uncertainties.umath as umath
from uncertainties import ufloat


# Effective permitivity rough estimate
def _e_e(e_r,h_s,w_r):
    return (1/2)*(e_r+1 + (e_r-1)*(1/umath.sqrt(1+12*h_s/w_r)))

# sapphire (at 20mK?)
e_r = ufloat(9.4,.1)

# height of sapphire (in meters) 1.5mm
h_s =  0.0015

# width of stripline resonator ?? not listed in paper 100micro m?
w_r = ufloat(0.0001,0.00005)

e_e = _e_e(e_r,h_s,w_r)

print(f"Permitivity of Sapphire: {e_r}\nEffective permitivity in stripline resonator system: {e_e}")
# does not seem correct.
e_e_2 = 2.4 


Permitivity of Sapphire: 9.40+/-0.10
Effective permitivity in stripline resonator system: 5.51+/-0.09


In [10]:
# stripline length 12mm
l = 0.012 
C_LIGHT   = 2.99792458e8    # m/s
f_1 = C_LIGHT/(2*l*umath.sqrt(e_e)) # Hz
f_1 = f_1 * 0.000001 # MHz
print(f"f_1 from e_e: {f_1:} MHz")

f_1_1 = C_LIGHT/(2*l*umath.sqrt(e_e_2)) # Hz
f_1_1 = f_1_1 * 0.000001 # MHz
print(f"f_1_1 from e_e_2: {f_1_1:} MHz")

f_1 from e_e: (5.32+/-0.05)e+03 MHz
f_1_1 from e_e_2: 8063.133313559627 MHz


### Derivations

#### Constants

In [3]:
"""
================================================================================
STRIPLINE-IN-CYLINDER RESONATOR: ANALYTIC APPROXIMATIONS
================================================================================

Physics: A thin superconducting strip (width w, length ℓ) suspended on a
sapphire substrate inside a superconducting cylindrical waveguide (radius R,
length ℓ_cyl), closed at both ends. The strip acts as the center conductor
of a coaxial-like TEM transmission line resonator with open-open boundary
conditions.

Architecture: Axline et al., APL 109, 042601 (2016) — "coaxial tunnel"
Follow-on: Ganjam et al. (2023), Krayzman et al. (2024)

References for each formula are noted inline.

All SI units unless noted.
================================================================================
"""

import numpy as np
import warnings
from dataclasses import dataclass, field
from typing import Optional, Tuple, List

# =============================================================================
# Physical constants
# =============================================================================
MU_0      = 4 * np.pi * 1e-7  # H/m
EPS_0     = 8.854187817e-12    # F/m
K_B       = 1.380649e-23       # J/K
H_PLANCK  = 6.62607015e-34    # J·s
HBAR      = H_PLANCK / (2 * np.pi)

# TE_11 first zero of J_1' (Bessel derivative)
X_PRIME_11 = 1.8411838         # Abramowitz & Stegun, Table 9.5
# TM_01 first zero of J_0
X_01       = 2.4048256



#### Design functions

In [4]:

@dataclass
class DesignParams:
    """
    All geometric and material parameters for one stripline resonator
    in a cylindrical tunnel enclosure.

    Geometric parameters:
        R           : inner radius of cylinder [m]
        ell         : strip length [m]
        ell_cyl     : cylinder (tunnel) length [m]
        w           : strip width [m]
        t           : strip thickness [m] (for effective-width correction)
        h_sub       : substrate thickness [m] (sapphire chip)
        eps_r_sub   : relative permittivity of substrate (sapphire ≈ 10)

    Material / operating parameters:
        T           : operating temperature [K]
        T_c         : superconductor critical temperature [K]
        R_s_residual: residual surface resistance [Ω]
        eps_eff     : effective dielectric constant (override; computed if None)

    Positioning:
        strip_offset: axial offset of strip center from cylinder center [m]
                      (0 = centered; nonzero if strip is not centered)
    """
    # Geometry
    R:            float = 1.25e-3     # 2.5 mm diameter tunnel
    ell:          float = 12.0e-3     # 12 mm strip
    ell_cyl:      float = 34.0e-3     # 34 mm tunnel
    w:            float = 100e-6      # 100 µm strip width
    t:            float = 200e-9      # 200 nm film thickness
    h_sub:        float = 430e-6      # 430 µm sapphire
    eps_r_sub:    float = 10.0        # sapphire c-axis, ~10 at microwave/cryo

    # Material
    T:            float = 20e-3       # 20 mK
    T_c:          float = 1.2         # Al T_c
    R_s_residual: float = 5e-9        # 5 nΩ residual

    # Override
    eps_eff:      Optional[float] = None  # if None, estimated from geometry

    # Positioning
    strip_offset: float = 0.0        # centered by default


# =============================================================================
# Approximation validity checks
# =============================================================================

def check_design_validity(p: DesignParams) -> List[str]:
    """
    Check whether the analytic approximations are valid for the given
    design parameters. Returns a list of warning strings.

    Checks implemented:
    1. w << R  (flat-stripline surrogate validity)
    2. Enclosure diameter not too large (higher-order mode suppression)
    3. Strip fits inside cylinder with adequate gaps
    4. Operating frequency well below cutoff
    5. Substrate thickness vs enclosure diameter
    6. Effective-width formula regime
    7. Temperature well below T_c
    """
    warns = []
    d = 2 * p.R  # diameter

    # --- 1. Strip width vs radius ---
    # The flat-stripline-to-cylindrical mapping requires w << 2R.
    # For w/2R > ~0.3, curvature corrections become large and the
    # uniform-current Q approximation is poor.
    ratio_w_2R = p.w / (2 * p.R)
    if ratio_w_2R > 0.3:
        warns.append(
            f"CRITICAL: w/(2R) = {ratio_w_2R:.3f} > 0.3. "
            f"Flat-stripline approximation for Z₀ is unreliable. "
            f"Current crowding invalidates uniform-current Q formula."
        )
    elif ratio_w_2R > 0.1:
        warns.append(
            f"WARNING: w/(2R) = {ratio_w_2R:.3f} > 0.1. "
            f"Curvature correction to Z₀ may be ~10-20%. "
            f"Use HFSS 2D port solve for reliable Z₀."
        )

    # --- 2. Enclosure diameter ---
    # Axline: tunnel diameters explored from ~1-6 mm.
    # Ganjam: Q_i improved with diameter up to ~4 mm, then degraded.
    # For d > ~8 mm, the cutoff frequency drops below ~30 GHz and
    # higher-order waveguide modes can couple to stripline modes,
    # breaking the clean TEM picture.
    # For d > 15 mm, TE₁₁ cutoff drops to ~12 GHz — likely within
    # the operating band's harmonics.
    d_mm = d * 1e3
    if d_mm > 15:
        warns.append(
            f"CRITICAL: Enclosure diameter = {d_mm:.1f} mm > 15 mm. "
            f"TE₁₁ cutoff = {cutoff_frequency(p.R)/1e9:.1f} GHz — "
            f"harmonics of stripline modes will overlap waveguide modes. "
            f"TEM transmission-line model is invalid."
        )
    elif d_mm > 8:
        warns.append(
            f"WARNING: Enclosure diameter = {d_mm:.1f} mm > 8 mm. "
            f"TE₁₁ cutoff = {cutoff_frequency(p.R)/1e9:.1f} GHz. "
            f"Second/third harmonics may approach cutoff. "
            f"Check mode spectrum carefully in HFSS."
        )
    elif d_mm > 6:
        warns.append(
            f"CAUTION: Enclosure diameter = {d_mm:.1f} mm > 6 mm. "
            f"Ganjam observed Q degradation for diameters above ~4 mm "
            f"(increased participation in package surfaces). "
            f"TEM model still valid but Q estimates are optimistic."
        )

    # --- 3. Strip fits in cylinder with adequate gaps ---
    g_each = gap_length(p)
    if g_each < 0:
        warns.append(
            f"CRITICAL: Strip length ({p.ell*1e3:.1f} mm) exceeds "
            f"cylinder length ({p.ell_cyl*1e3:.1f} mm). Unphysical."
        )
    else:
        gamma_val = evanescent_gamma(p.R, 5e9)  # rough estimate at 5 GHz
        n_decay = g_each * gamma_val
        if n_decay < 3:
            warns.append(
                f"WARNING: Gap g = {g_each*1e3:.2f} mm ≈ {n_decay:.1f} × γ⁻¹. "
                f"End correction model assumes g >> γ⁻¹. "
                f"With < 3 decay lengths, the closed end wall perturbs "
                f"the fringing fields and Δℓ estimate is unreliable. "
                f"Use HFSS eigenmode to extract true f."
            )
        if g_each < 2e-3:
            warns.append(
                f"WARNING: Gap g = {g_each*1e3:.2f} mm < 2 mm. "
                f"Seam loss from end wall proximity may be significant. "
                f"Axline recommends ≥ 7 mm from waveguide end for Q_i ≥ 10⁸."
            )

    # --- 4. Substrate thickness ---
    # If h_sub > R, the chip doesn't physically fit.
    # If h_sub > 0.5 * 2R, the quasi-stripline picture is very approximate.
    if p.h_sub > 2 * p.R:
        warns.append(
            f"CRITICAL: Substrate thickness ({p.h_sub*1e6:.0f} µm) > "
            f"enclosure diameter ({2*p.R*1e3:.2f} mm). Unphysical."
        )
    elif p.h_sub > 0.5 * 2 * p.R:
        warns.append(
            f"WARNING: Substrate fills > 50% of enclosure diameter. "
            f"Quasi-stripline field picture is poor; most field energy "
            f"is in sapphire. ε_eff → ε_r."
        )

    # --- 5. Effective-width formula regime (Pozar eq. 3.179) ---
    # Pozar's W_e correction is: W_e/b = W/b - (0.35 - W/b)² for W/b < 0.35
    # This gives W_e < W (narrower effective width), which is fine for
    # moderate w/b. But for very small w/b (< ~0.05), the correction
    # produces W_e that may be unphysically small or even negative for
    # the enclosed-stripline formula. Pozar's formula was fitted for
    # w/b ~ 0.1-0.9.
    if ratio_w_2R < 0.05:
        warns.append(
            f"CAUTION: w/(2R) = {ratio_w_2R:.4f} < 0.05. "
            f"Pozar's effective-width correction is outside its fitted range. "
            f"Using W_e ≈ w (no correction) is safer for very narrow strips. "
            f"Z₀ estimate may be off by ~20%."
        )

    # --- 6. Temperature ---
    if p.T > 0.3 * p.T_c:
        warns.append(
            f"WARNING: T/T_c = {p.T/p.T_c:.2f} > 0.3. "
            f"BCS surface resistance is no longer exponentially suppressed. "
            f"Conductor loss may become relevant."
        )

    # --- 7. ε_eff sanity ---
    if p.eps_eff is not None:
        if p.eps_eff < 1.0:
            warns.append(
                f"CRITICAL: ε_eff = {p.eps_eff:.2f} < 1. Unphysical."
            )
        if p.eps_eff > p.eps_r_sub:
            warns.append(
                f"WARNING: ε_eff = {p.eps_eff:.2f} > ε_r(substrate) = "
                f"{p.eps_r_sub:.1f}. This implies more field in dielectric "
                f"than a fully-filled geometry. Check value."
            )

    return warns



#### Core electromagnetic functions


In [5]:
def cutoff_frequency(R: float, mode: str = "TE11") -> float:
    """
    Cutoff frequency of circular waveguide (no center conductor).

    TE_11: f_c = c * x'_{11} / (2π R),  x'_{11} ≈ 1.8412
    TM_01: f_c = c * x_{01} / (2π R),   x_{01} ≈ 2.4048

    Args:
        R: inner radius [m]
        mode: "TE11" or "TM01"

    Returns:
        f_c [Hz]
    """
    if mode == "TE11":
        x = X_PRIME_11
    elif mode == "TM01":
        x = X_01
    else:
        raise ValueError(f"Unknown mode: {mode}")
    return C_LIGHT * x / (2 * np.pi * R)


def evanescent_gamma(R: float, f: float, mode: str = "TE11") -> float:
    """
    Evanescent decay constant γ in the beyond-strip cylindrical waveguide
    sections (no center conductor).

    γ = sqrt(k_c² - k²),  k = 2πf/c,  k_c = x'_{11}/R

    Returns γ [1/m]. Returns 0 if f > f_c (propagating).
    """
    k = 2 * np.pi * f / C_LIGHT
    if mode == "TE11":
        k_c = X_PRIME_11 / R
    elif mode == "TM01":
        k_c = X_01 / R
    else:
        raise ValueError(f"Unknown mode: {mode}")

    if k >= k_c:
        return 0.0  # propagating, not evanescent
    return np.sqrt(k_c**2 - k**2)


def gap_length(p: DesignParams) -> float:
    """
    Gap between each strip end and the nearest cylinder end wall.
    Assumes strip is centered (or offset by strip_offset).

    g = (ℓ_cyl - ℓ) / 2  (if centered)
    """
    g = (p.ell_cyl - p.ell) / 2
    # If offset, one gap shrinks, other grows
    # g_near = g - |offset|, g_far = g + |offset|
    # Return the smaller gap (limiting case)
    g_near = g - abs(p.strip_offset)
    return g_near


def gap_lengths_both(p: DesignParams) -> Tuple[float, float]:
    """Returns (g_near, g_far) gap lengths accounting for strip offset."""
    g = (p.ell_cyl - p.ell) / 2
    g1 = g - p.strip_offset  # one end
    g2 = g + p.strip_offset  # other end
    return (g1, g2)


# =============================================================================
# Characteristic impedance
# =============================================================================

def effective_width(w: float, b: float) -> float:
    """
    Pozar eq. 3.179b: effective strip width for enclosed stripline.

    W_e/b = W/b                         if W/b > 0.35
    W_e/b = W/b - (0.35 - W/b)²        if W/b < 0.35

    For very narrow strips (W/b < 0.05), this correction is outside
    Pozar's fitted range. We clamp W_e >= w/2 to avoid unphysical values.

    Args:
        w: physical strip width [m]
        b: ground-plane spacing [m] (= 2R for cylindrical surrogate)

    Returns:
        w_e: effective width [m]
    """
    ratio = w / b
    if ratio >= 0.35:
        w_e = w
    else:
        w_e_over_b = ratio - (0.35 - ratio)**2
        w_e = w_e_over_b * b
        # Clamp: effective width shouldn't go below half the physical width
        w_e = max(w_e, w * 0.5)
    return w_e


def Z0_flat_stripline(w: float, R: float, eps_r: float = 1.0,
                       use_we_correction: bool = True) -> float:
    """
    Characteristic impedance of stripline using flat-ground-plane
    approximation with b = 2R.

    Z₀ = (30π / √ε_r) / (W_e/(2R) + 0.441)

    Ref: Pozar, Microwave Engineering 4e, eq. (3.179)

    This is a ZEROTH-ORDER estimate. The cylindrical wall curvature
    increases capacitance (decreases Z₀) by O((w/R)²).

    Args:
        w: strip width [m]
        R: cylinder radius [m]
        eps_r: relative permittivity filling the space (1 for vacuum)
        use_we_correction: apply Pozar's effective-width correction

    Returns:
        Z₀ [Ω]
    """
    b = 2 * R
    if use_we_correction:
        w_e = effective_width(w, b)
    else:
        w_e = w
    return 30 * np.pi / (np.sqrt(eps_r) * (w_e / b + 0.441))


def Z0_coax_equivalent(w: float, R: float) -> float:
    """
    Sanity-check impedance: treat strip as equivalent round conductor
    of radius a_eff ≈ w/4 in a coaxial line of outer radius R.

    Z₀ = 60 ln(R / a_eff)

    This typically gives Z₀ ~15% higher than the stripline estimate
    (strip has more capacitance than an equivalent round wire).

    Args:
        w: strip width [m]
        R: cylinder radius [m]

    Returns:
        Z₀ [Ω]
    """
    a_eff = w / 4
    if a_eff <= 0 or R <= a_eff:
        return np.nan
    return 60 * np.log(R / a_eff)


def Z0_with_dielectric(Z0_vac: float, eps_eff: float) -> float:
    """
    Impedance corrected for effective dielectric constant.

    Z₀ = Z₀(vac) / √ε_eff

    Args:
        Z0_vac: vacuum impedance [Ω]
        eps_eff: effective dielectric constant

    Returns:
        Z₀ [Ω]
    """
    return Z0_vac / np.sqrt(eps_eff)


def estimate_eps_eff(h_sub: float, R: float, eps_r_sub: float) -> float:
    """
    Rough estimate of effective dielectric constant for a substrate
    of thickness h_sub centered in a cylinder of radius R.

    Uses a parallel-capacitor model:
        ε_eff ≈ 1 + (ε_r - 1) * (h_sub / (2R))

    This is a CRUDE first-order estimate. The real value depends
    sensitively on strip width, chip width, and field distribution.
    HFSS extraction is strongly recommended.

    For Axline-type devices (sapphire ε_r ≈ 10, h_sub ~ 430 µm,
    d ~ 2.5-5 mm), typical ε_eff ~ 4-6.

    Args:
        h_sub: substrate thickness [m]
        R: cylinder radius [m]
        eps_r_sub: substrate relative permittivity

    Returns:
        ε_eff (dimensionless)
    """
    filling_fraction = h_sub / (2 * R)
    filling_fraction = min(filling_fraction, 1.0)  # can't exceed full fill
    eps_eff = 1 + (eps_r_sub - 1) * filling_fraction
    return eps_eff


def capacitance_per_length(Z0: float, eps_eff: float = 1.0) -> float:
    """
    Capacitance per unit length from Z₀ and phase velocity.

    C' = 1/(v_p Z₀) = √ε_eff / (c Z₀)

    Args:
        Z0: characteristic impedance [Ω] (already including ε_eff if applicable)
        eps_eff: effective dielectric constant (only needed if Z0 is vacuum value)

    Returns:
        C' [F/m]
    """
    v_p = C_LIGHT / np.sqrt(eps_eff)
    return 1.0 / (v_p * Z0)



#### End correction and eigenfrequencies


In [6]:

def end_correction_delta_ell(R: float, f: float, g: float,
                              mode: str = "TE11") -> float:
    """
    End-correction length extension for one open end of the strip,
    modeled as evanescent energy storage in the beyond-strip cylindrical
    waveguide section.

    Δℓ = (1 - exp(-2γg)) / (2γ)

    For g >> γ⁻¹:  Δℓ → 1/(2γ)
    For g → 0:     Δℓ → 0

    Derivation: integrate evanescent electric energy density ∝ exp(-2γz)
    over the gap, equate to lumped end capacitance C_end = C' × Δℓ / (v_p Z₀).
    See analysis document, §2.4.

    This uses the TE₁₁ mode (lowest cutoff) for a conservative (largest)
    estimate of Δℓ. TM₀₁ has higher cutoff and gives smaller Δℓ.

    Args:
        R: cylinder radius [m]
        f: frequency [Hz] (self-consistent iteration may be needed)
        g: gap length from strip end to cylinder end wall [m]
        mode: waveguide mode for evanescent decay

    Returns:
        Δℓ [m]
    """
    gamma = evanescent_gamma(R, f, mode)
    if gamma <= 0:
        # Above cutoff — evanescent model invalid
        warnings.warn(
            f"f = {f/1e9:.2f} GHz is above {mode} cutoff "
            f"({cutoff_frequency(R, mode)/1e9:.1f} GHz). "
            f"End correction is undefined; mode is propagating in gap."
        )
        return np.nan

    if g <= 0:
        return 0.0

    exp_term = np.exp(-2 * gamma * g)
    return (1 - exp_term) / (2 * gamma)


def eigenfrequencies(p: DesignParams, n_modes: int = 5,
                     eps_eff: Optional[float] = None,
                     include_end_correction: bool = True,
                     iterate_end_correction: bool = True,
                     max_iter: int = 10, tol: float = 1e-6
                     ) -> dict:
    """
    Compute TEM eigenmode frequencies for the open-open strip resonator.

    f_n = n * c / (2 * ℓ_eff * √ε_eff)

    where ℓ_eff = ℓ + Δℓ₁ + Δℓ₂  (one correction per open end).

    The end correction Δℓ depends on f (through γ), so this is solved
    self-consistently by iteration if iterate_end_correction=True.

    Args:
        p: DesignParams
        n_modes: number of modes to compute (n = 1, 2, ..., n_modes)
        eps_eff: override effective dielectric constant
        include_end_correction: whether to include Δℓ
        iterate_end_correction: iterate for self-consistency
        max_iter: max iterations for self-consistency
        tol: relative frequency convergence tolerance

    Returns:
        dict with keys:
            'n': mode indices [array]
            'f_Hz': frequencies [Hz, array]
            'f_GHz': frequencies [GHz, array]
            'ell_eff': effective length [m]
            'delta_ell_1': end correction at end 1 [m]
            'delta_ell_2': end correction at end 2 [m]
            'eps_eff': effective dielectric constant used
            'warnings': list of warning strings
    """
    if eps_eff is None:
        if p.eps_eff is not None:
            eps_eff = p.eps_eff
        else:
            eps_eff = 1.0  # vacuum

    g1, g2 = gap_lengths_both(p)
    ns = np.arange(1, n_modes + 1)
    mode_warnings = []

    if not include_end_correction:
        ell_eff = p.ell
        dl1, dl2 = 0.0, 0.0
        f_n = ns * C_LIGHT / (2 * ell_eff * np.sqrt(eps_eff))
    else:
        # Initial guess: no end correction
        f_n = ns * C_LIGHT / (2 * p.ell * np.sqrt(eps_eff))

        for iteration in range(max_iter):
            # Compute end corrections at each frequency
            # Use fundamental frequency for Δℓ (weak dependence on n)
            f_ref = f_n[0]
            dl1 = end_correction_delta_ell(p.R, f_ref, g1)
            dl2 = end_correction_delta_ell(p.R, f_ref, g2)
            if np.isnan(dl1) or np.isnan(dl2):
                mode_warnings.append(
                    "End correction returned NaN — frequency above cutoff?"
                )
                break

            ell_eff = p.ell + dl1 + dl2
            f_n_new = ns * C_LIGHT / (2 * ell_eff * np.sqrt(eps_eff))

            if not iterate_end_correction:
                f_n = f_n_new
                break

            # Check convergence
            rel_change = np.max(np.abs(f_n_new - f_n) / f_n)
            f_n = f_n_new
            if rel_change < tol:
                break
        else:
            mode_warnings.append(
                f"End-correction iteration did not converge after {max_iter} steps."
            )

    # Check if any harmonics approach cutoff
    f_c = cutoff_frequency(p.R)
    for i, f in enumerate(f_n):
        if f > 0.5 * f_c:
            mode_warnings.append(
                f"Mode n={ns[i]}: f = {f/1e9:.2f} GHz > 0.5 × f_c = "
                f"{0.5*f_c/1e9:.1f} GHz. Higher-order waveguide modes "
                f"may hybridize with this TEM harmonic."
            )
        if f > f_c:
            mode_warnings.append(
                f"Mode n={ns[i]}: f = {f/1e9:.2f} GHz EXCEEDS cutoff "
                f"f_c = {f_c/1e9:.1f} GHz. This mode is NOT a "
                f"confined TEM resonance."
            )

    return {
        'n': ns,
        'f_Hz': f_n,
        'f_GHz': f_n / 1e9,
        'ell_eff': p.ell + dl1 + dl2 if include_end_correction else p.ell,
        'delta_ell_1': dl1 if include_end_correction else 0.0,
        'delta_ell_2': dl2 if include_end_correction else 0.0,
        'eps_eff': eps_eff,
        'warnings': mode_warnings,
    }



#### Quality factor


In [7]:

def bcs_surface_resistance(f: float, T: float, T_c: float,
                           Delta_ratio: float = 1.76) -> float:
    """
    BCS surface resistance (approximate, Mattis-Bardeen low-T limit).

    R_s^BCS ∝ (ω² / T) exp(-Δ / k_B T)

    The proportionality constant depends on London penetration depth,
    normal-state conductivity, etc. For aluminum at GHz frequencies:
    R_s^BCS ~ A × f² × exp(-Δ/kT) / T

    At T = 20 mK, T_c = 1.2 K, f = 5 GHz:
    Δ/(k_B T) ≈ 1.76 × 1.2 / 0.020 = 105.6
    exp(-105.6) ~ 10⁻⁴⁶

    This is astronomically small — thermal quasiparticle loss is
    completely negligible at dilution fridge temperatures for Al.

    Returns:
        R_s^BCS [Ω] (will be ~0 for T << T_c)
    """
    Delta_0 = Delta_ratio * K_B * T_c   # BCS gap energy
    exponent = -Delta_0 / (K_B * T)

    # Prefactor: use approximate Mattis-Bardeen result for Al
    # A ~ µ₀² ω² λ³ σ_n ~ 10⁻²⁴ Ω for typical Al parameters at 5 GHz
    # This prefactor is geometry/material-dependent; for order-of-magnitude:
    omega = 2 * np.pi * f
    # Approximate prefactor from Gao thesis Ch.2
    A = 1e-24  # rough for Al, λ_L ~ 50 nm, σ_n ~ 3.7e7 S/m
    R_s = A * omega**2 / T * np.exp(exponent)
    return R_s


def total_surface_resistance(f: float, T: float, T_c: float,
                              R_s_residual: float = 5e-9) -> float:
    """
    Total surface resistance: R_s = R_s^BCS + R_s^residual.

    At dilution fridge temperatures for Al, R_s ≈ R_s^residual.

    Args:
        f: frequency [Hz]
        T: temperature [K]
        T_c: critical temperature [K]
        R_s_residual: residual surface resistance [Ω]

    Returns:
        R_s [Ω]
    """
    R_bcs = bcs_surface_resistance(f, T, T_c)
    return R_bcs + R_s_residual


def conductor_attenuation(Z0: float, w: float, R: float,
                           R_s: float) -> float:
    """
    Attenuation constant from conductor loss, uniform-current approximation.

    α_c = R_s / (4 Z₀) × (1/w + 1/(πR))

    Ref: power-loss method (Pozar §2.7), adapted for strip-in-cylinder.
    See analysis document §3.2.

    NOTE: This has ~factor-of-2 uncertainty due to:
    - Current crowding at strip edges (increases loss)
    - Non-uniform current on cylinder wall (concentrated near strip)
    - Wheeler's incremental inductance rule gives different geometric factor

    Args:
        Z0: characteristic impedance [Ω]
        w: strip width [m]
        R: cylinder radius [m]
        R_s: surface resistance [Ω]

    Returns:
        α_c [Np/m]
    """
    return R_s / (4 * Z0) * (1/w + 1/(np.pi * R))


def Q_conductor(beta: float, Z0: float, w: float, R: float,
                R_s: float) -> float:
    """
    Unloaded Q from conductor loss only.

    Q₀ = β / (2α_c) = 2β Z₀ / [R_s (1/w + 1/(πR))]

    For the n-th mode: β_n = nπ / ℓ_eff

    Args:
        beta: propagation constant 2πf/v_p [rad/m]
        Z0: characteristic impedance [Ω]
        w: strip width [m]
        R: cylinder radius [m]
        R_s: surface resistance [Ω]

    Returns:
        Q₀ (dimensionless)
    """
    alpha_c = conductor_attenuation(Z0, w, R, R_s)
    if alpha_c <= 0:
        return np.inf
    return beta / (2 * alpha_c)


def Q_radiation(R: float, f: float, g1: float, g2: float) -> float:
    """
    Order-of-magnitude radiation Q from evanescent leakage through
    the beyond-strip cylindrical sections.

    Q_rad ~ exp(2γg) (very rough scaling)

    Both ends contribute: 1/Q_rad ≈ 1/Q_rad1 + 1/Q_rad2

    This is NOT a precise formula — a mode-matching calculation is
    needed for the prefactor. But the exponential scaling shows whether
    radiation is relevant.

    Args:
        R: cylinder radius [m]
        f: frequency [Hz]
        g1, g2: gap lengths at each end [m]

    Returns:
        Q_rad (dimensionless, order of magnitude)
    """
    gamma = evanescent_gamma(R, f)
    if gamma <= 0:
        return 0.0  # above cutoff, no confinement
    # Each end contributes; take geometric mean as rough estimate
    Q1 = np.exp(2 * gamma * g1) if g1 > 0 else 1.0
    Q2 = np.exp(2 * gamma * g2) if g2 > 0 else 1.0
    # Combined (parallel)
    return 1.0 / (1.0/Q1 + 1.0/Q2)


def geometry_factor(beta: float, Z0: float, w: float, R: float) -> float:
    """
    Geometry factor G = Q₀ × R_s (material-independent figure of merit).

    G = 2β Z₀ / (1/w + 1/(πR))

    Note: for TEM-line resonators, G depends on frequency (through β),
    unlike the purely geometric G of 3D cavities.

    Args:
        beta: propagation constant [rad/m]
        Z0: characteristic impedance [Ω]
        w: strip width [m]
        R: cylinder radius [m]

    Returns:
        G [Ω]
    """
    return 2 * beta * Z0 / (1/w + 1/(np.pi * R))


def compute_Q_budget(p: DesignParams, f: float, Z0: float,
                      eps_eff: float, ell_eff: float,
                      tan_delta_bulk: float = 1e-8,
                      p_bulk: float = 0.3,
                      tan_delta_surf: float = 2e-3,
                      p_surf: float = 1e-4,
                      ) -> dict:
    """
    Compute Q contributions from all known loss channels.

    1/Q_total = 1/Q_cond + 1/Q_rad + 1/Q_bulk + 1/Q_surf

    Participation ratios (p_bulk, p_surf) are geometry-dependent and
    should ultimately come from HFSS field integration. Default values
    are order-of-magnitude estimates from Ganjam et al.

    Args:
        p: DesignParams
        f: frequency [Hz]
        Z0: characteristic impedance [Ω]
        eps_eff: effective dielectric constant
        ell_eff: effective resonator length [m]
        tan_delta_bulk: bulk dielectric loss tangent (sapphire ~ 10⁻⁸ at cryo)
        p_bulk: bulk dielectric participation ratio
        tan_delta_surf: surface dielectric loss tangent (~2×10⁻³)
        p_surf: surface participation ratio

    Returns:
        dict with individual and total Q values
    """
    v_p = C_LIGHT / np.sqrt(eps_eff)
    beta = 2 * np.pi * f / v_p

    R_s = total_surface_resistance(f, p.T, p.T_c, p.R_s_residual)
    Q_cond = Q_conductor(beta, Z0, p.w, p.R, R_s)

    g1, g2 = gap_lengths_both(p)
    Q_rad = Q_radiation(p.R, f, g1, g2)

    # Dielectric losses (participation-ratio model)
    Q_bulk_diel = 1.0 / (p_bulk * tan_delta_bulk) if (p_bulk * tan_delta_bulk) > 0 else np.inf
    Q_surf_diel = 1.0 / (p_surf * tan_delta_surf) if (p_surf * tan_delta_surf) > 0 else np.inf

    # Total
    inv_Q_total = (1/Q_cond + 1/Q_rad + 1/Q_bulk_diel + 1/Q_surf_diel)
    Q_total = 1.0 / inv_Q_total if inv_Q_total > 0 else np.inf

    G = geometry_factor(beta, Z0, p.w, p.R)

    return {
        'R_s': R_s,
        'alpha_c': conductor_attenuation(Z0, p.w, p.R, R_s),
        'Q_conductor': Q_cond,
        'Q_radiation': Q_rad,
        'Q_bulk_dielectric': Q_bulk_diel,
        'Q_surface_dielectric': Q_surf_diel,
        'Q_total': Q_total,
        'Q_limiting': min(Q_cond, Q_rad, Q_bulk_diel, Q_surf_diel),
        'limiting_mechanism': ['conductor', 'radiation', 'bulk_diel', 'surf_diel'][
            np.argmin([Q_cond, Q_rad, Q_bulk_diel, Q_surf_diel])
        ],
        'geometry_factor_G': G,
        'beta': beta,
    }



#### Multi-resonator: frequency planning for N strips in one tunnel


In [8]:

# =============================================================================
# Multi-resonator: frequency planning for N strips in one tunnel
# =============================================================================

def plan_multi_resonator(target_freqs_GHz: List[float],
                          p_template: DesignParams,
                          eps_eff: float,
                          min_gap_mm: float = 3.0,
                          ) -> dict:
    """
    Given target frequencies for N stripline resonators in one tunnel,
    compute required strip lengths and check if they fit.

    Args:
        target_freqs_GHz: list of target frequencies [GHz]
        p_template: template DesignParams (uses R, ell_cyl, etc.)
        eps_eff: effective dielectric constant
        min_gap_mm: minimum gap between strips and between strip/wall [mm]

    Returns:
        dict with strip lengths, total usage, feasibility
    """
    n_res = len(target_freqs_GHz)
    strip_lengths = []
    for f_GHz in target_freqs_GHz:
        f = f_GHz * 1e9
        # ℓ ≈ c / (2f √ε_eff) (ignoring end correction for initial estimate)
        ell = C_LIGHT / (2 * f * np.sqrt(eps_eff))
        strip_lengths.append(ell)

    total_strip = sum(strip_lengths)
    total_gaps = (n_res + 1) * min_gap_mm * 1e-3  # N+1 gaps (both ends + between)
    total_needed = total_strip + total_gaps
    fits = total_needed <= p_template.ell_cyl

    # Isolation between adjacent strips
    gamma_min = evanescent_gamma(p_template.R, max(target_freqs_GHz) * 1e9)
    isolation_dB = 20 * np.log10(np.e) * 2 * gamma_min * min_gap_mm * 1e-3

    return {
        'n_resonators': n_res,
        'target_freqs_GHz': target_freqs_GHz,
        'strip_lengths_mm': [l * 1e3 for l in strip_lengths],
        'total_strip_mm': total_strip * 1e3,
        'total_gaps_mm': total_gaps * 1e3,
        'total_needed_mm': total_needed * 1e3,
        'tunnel_length_mm': p_template.ell_cyl * 1e3,
        'fits': fits,
        'margin_mm': (p_template.ell_cyl - total_needed) * 1e3,
        'inter_strip_isolation_dB': isolation_dB,
    }


# =============================================================================
# Convenience: full single-resonator analysis
# =============================================================================

def full_analysis(p: DesignParams,
                   n_modes: int = 5,
                   include_end_correction: bool = True,
                   tan_delta_bulk: float = 1e-8,
                   p_bulk: float = 0.3,
                   tan_delta_surf: float = 2e-3,
                   p_surf: float = 1e-4,
                   verbose: bool = True,
                   ) -> dict:
    """
    Run complete analytic analysis for one stripline resonator.

    Returns a comprehensive dict with all computed quantities.
    Prints a formatted report if verbose=True.
    """
    results = {}

    # --- Validity checks ---
    validity_warnings = check_design_validity(p)
    results['validity_warnings'] = validity_warnings

    # --- Effective dielectric constant ---
    if p.eps_eff is not None:
        eps_eff = p.eps_eff
        eps_eff_source = "user override"
    else:
        eps_eff = estimate_eps_eff(p.h_sub, p.R, p.eps_r_sub)
        eps_eff_source = "parallel-capacitor estimate (crude)"
    results['eps_eff'] = eps_eff
    results['eps_eff_source'] = eps_eff_source

    # --- Cutoff ---
    f_c_TE11 = cutoff_frequency(p.R, "TE11")
    f_c_TM01 = cutoff_frequency(p.R, "TM01")
    results['f_c_TE11_GHz'] = f_c_TE11 / 1e9
    results['f_c_TM01_GHz'] = f_c_TM01 / 1e9

    # --- Gaps and evanescent properties ---
    g1, g2 = gap_lengths_both(p)
    results['g1_mm'] = g1 * 1e3
    results['g2_mm'] = g2 * 1e3

    # --- Impedance ---
    Z0_vac = Z0_flat_stripline(p.w, p.R, eps_r=1.0)
    Z0_vac_no_we = Z0_flat_stripline(p.w, p.R, eps_r=1.0, use_we_correction=False)
    Z0_coax = Z0_coax_equivalent(p.w, p.R)
    Z0_eff = Z0_with_dielectric(Z0_vac, eps_eff)
    results['Z0_vac'] = Z0_vac
    results['Z0_vac_no_we_correction'] = Z0_vac_no_we
    results['Z0_coax_equiv'] = Z0_coax
    results['Z0_with_eps_eff'] = Z0_eff

    # --- Eigenfrequencies: vacuum ---
    modes_vac = eigenfrequencies(p, n_modes, eps_eff=1.0,
                                  include_end_correction=include_end_correction)
    results['modes_vacuum'] = modes_vac

    # --- Eigenfrequencies: with dielectric ---
    modes_diel = eigenfrequencies(p, n_modes, eps_eff=eps_eff,
                                   include_end_correction=include_end_correction)
    results['modes_with_dielectric'] = modes_diel

    # --- Eigenfrequencies: no end correction (for comparison) ---
    modes_no_ec = eigenfrequencies(p, n_modes, eps_eff=eps_eff,
                                    include_end_correction=False)
    results['modes_no_end_correction'] = modes_no_ec

    # --- Q budget at fundamental ---
    f1 = modes_diel['f_Hz'][0]
    ell_eff = modes_diel['ell_eff']
    Q_budget = compute_Q_budget(
        p, f1, Z0_eff, eps_eff, ell_eff,
        tan_delta_bulk=tan_delta_bulk, p_bulk=p_bulk,
        tan_delta_surf=tan_delta_surf, p_surf=p_surf,
    )
    results['Q_budget'] = Q_budget

    # --- Evanescent properties at fundamental ---
    gamma_f1 = evanescent_gamma(p.R, f1)
    results['gamma_at_f1'] = gamma_f1
    results['decay_length_mm'] = 1e3 / gamma_f1 if gamma_f1 > 0 else np.inf
    results['gamma_g1'] = gamma_f1 * g1
    results['gamma_g2'] = gamma_f1 * g2

    # --- Print report ---
    if verbose:
        print("=" * 72)
        print("STRIPLINE-IN-CYLINDER RESONATOR: ANALYTIC ESTIMATES")
        print("=" * 72)

        if validity_warnings:
            print("\n⚠  VALIDITY WARNINGS:")
            for w in validity_warnings:
                print(f"   • {w}")
            print()

        print(f"  Enclosure:  R = {p.R*1e3:.3f} mm  (d = {2*p.R*1e3:.2f} mm)")
        print(f"  Strip:      ℓ = {p.ell*1e3:.2f} mm,  w = {p.w*1e6:.1f} µm")
        print(f"  Tunnel:     ℓ_cyl = {p.ell_cyl*1e3:.1f} mm")
        print(f"  Substrate:  h = {p.h_sub*1e6:.0f} µm,  ε_r = {p.eps_r_sub:.1f}")
        print(f"  Gaps:       g₁ = {g1*1e3:.2f} mm,  g₂ = {g2*1e3:.2f} mm")

        print(f"\n--- Waveguide cutoff ---")
        print(f"  TE₁₁: {f_c_TE11/1e9:.1f} GHz")
        print(f"  TM₀₁: {f_c_TM01/1e9:.1f} GHz")
        print(f"  γ⁻¹ at f₁: {results['decay_length_mm']:.3f} mm")
        print(f"  γ×g₁ = {results['gamma_g1']:.1f}  →  "
              f"e⁻²ᵞᵍ = {np.exp(-2*gamma_f1*g1):.1e}")

        print(f"\n--- Characteristic impedance ---")
        print(f"  Z₀ (vacuum, flat stripline): {Z0_vac:.1f} Ω")
        print(f"  Z₀ (vacuum, coax equiv):     {Z0_coax:.1f} Ω")
        print(f"  ε_eff = {eps_eff:.2f}  ({eps_eff_source})")
        print(f"  Z₀ (with ε_eff):             {Z0_eff:.1f} Ω")

        print(f"\n--- Eigenfrequencies (first {n_modes} modes) ---")
        print(f"  {'n':>3}  {'vacuum':>10}  {'no Δℓ':>10}  {'with Δℓ':>10}")
        for i in range(n_modes):
            print(f"  {modes_vac['n'][i]:3d}  "
                  f"{modes_vac['f_GHz'][i]:8.3f} GHz  "
                  f"{modes_no_ec['f_GHz'][i]:8.3f} GHz  "
                  f"{modes_diel['f_GHz'][i]:8.3f} GHz")
        print(f"  End correction: Δℓ₁ = {modes_diel['delta_ell_1']*1e3:.3f} mm, "
              f"Δℓ₂ = {modes_diel['delta_ell_2']*1e3:.3f} mm")
        print(f"  ℓ_eff = {modes_diel['ell_eff']*1e3:.3f} mm")

        if modes_diel['warnings']:
            for w in modes_diel['warnings']:
                print(f"   ⚠ {w}")

        print(f"\n--- Q factor budget (fundamental, f₁ = {f1/1e9:.3f} GHz) ---")
        print(f"  R_s = {Q_budget['R_s']*1e9:.2f} nΩ  (residual-dominated)")
        print(f"  Q_conductor:  {Q_budget['Q_conductor']:.2e}")
        print(f"  Q_radiation:  {Q_budget['Q_radiation']:.2e}")
        print(f"  Q_bulk_diel:  {Q_budget['Q_bulk_dielectric']:.2e}")
        print(f"  Q_surf_diel:  {Q_budget['Q_surface_dielectric']:.2e}")
        print(f"  Q_total:      {Q_budget['Q_total']:.2e}")
        print(f"  Limiting:     {Q_budget['limiting_mechanism']}")
        print(f"  G (geom):     {Q_budget['geometry_factor_G']:.2f} Ω")

        print("\n" + "=" * 72)

    return results

### Application

####  SINGLE RESONATOR ANALYSIS — baseline parameters

In [9]:
"""
================================================================================
APPLICATION: Axline coaxial-tunnel stripline resonator analysis
================================================================================

Reproduces the back-of-envelope estimates from the analysis document,
using the derivations module above.

Design target: single stripline resonator in a cylindrical Al tunnel,
matching the Axline et al. APL 109, 042601 (2016) architecture.

Parameters from the analysis:
    Strip length:       12 mm
    Tunnel length:      34 mm
    Tunnel diameter:    2.5 mm  (R = 1.25 mm)
    Strip width:        100 µm  (representative; not specified by user)
    Substrate:          430 µm c-plane sapphire, ε_r ≈ 10
    Metal:              aluminum, T_c = 1.2 K
    Operating temp:     20 mK
================================================================================
"""

# If running in a notebook, the derivations code block above would be in
# a prior cell. If running as a script, import it:
# from derivations import *


print("\n" + "█" * 72)
print("  PART 1: SINGLE RESONATOR — AXLINE ARCHITECTURE")
print("█" * 72 + "\n")

params = DesignParams(
    R          = 1.25e-3,     # 2.5 mm diameter
    ell        = 12.0e-3,     # 12 mm strip
    ell_cyl    = 34.0e-3,     # 34 mm tunnel
    w          = 100e-6,      # 100 µm strip width
    t          = 200e-9,      # 200 nm Al film
    h_sub      = 430e-6,      # 430 µm sapphire
    eps_r_sub  = 10.0,        # sapphire
    T          = 20e-3,       # 20 mK
    T_c        = 1.2,         # Al
    R_s_residual = 5e-9,      # 5 nΩ residual
    eps_eff    = None,         # let the code estimate
)

results_auto_eps = full_analysis(
    params,
    n_modes=5,
    include_end_correction=True,
    # Ganjam-scale participation ratios (order of magnitude)
    tan_delta_bulk=1e-8,    # sapphire bulk loss tangent at cryo
    p_bulk=0.3,             # ~30% field energy in sapphire (geometry-dependent)
    tan_delta_surf=2e-3,    # surface TLS loss tangent
    p_surf=1e-4,            # ~10⁻⁴ surface participation
    verbose=True,
)



████████████████████████████████████████████████████████████████████████
  PART 1: SINGLE RESONATOR — AXLINE ARCHITECTURE
████████████████████████████████████████████████████████████████████████

STRIPLINE-IN-CYLINDER RESONATOR: ANALYTIC ESTIMATES

⚠  VALIDITY WARNINGS:
   • CAUTION: w/(2R) = 0.0400 < 0.05. Pozar's effective-width correction is outside its fitted range. Using W_e ≈ w (no correction) is safer for very narrow strips. Z₀ estimate may be off by ~20%.

  Enclosure:  R = 1.250 mm  (d = 2.50 mm)
  Strip:      ℓ = 12.00 mm,  w = 100.0 µm
  Tunnel:     ℓ_cyl = 34.0 mm
  Substrate:  h = 430 µm,  ε_r = 10.0
  Gaps:       g₁ = 11.00 mm,  g₂ = 11.00 mm

--- Waveguide cutoff ---
  TE₁₁: 70.3 GHz
  TM₀₁: 91.8 GHz
  γ⁻¹ at f₁: 0.683 mm
  γ×g₁ = 16.1  →  e⁻²ᵞᵍ = 1.0e-14

--- Characteristic impedance ---
  Z₀ (vacuum, flat stripline): 204.4 Ω
  Z₀ (vacuum, coax equiv):     234.7 Ω
  ε_eff = 2.55  (parallel-capacitor estimate (crude))
  Z₀ (with ε_eff):             128.1 Ω

--- Eigenfre

#### 2. COMPARISON: different ε_eff values


In [10]:


# The auto-estimated ε_eff from the parallel-capacitor model is crude.
# Here we show how f₁ varies with ε_eff to calibrate against HFSS.

print("\n" + "█" * 72)
print("  PART 2: FREQUENCY vs ε_eff (FOR HFSS CALIBRATION)")
print("█" * 72 + "\n")

print(f"  Strip length ℓ = {params.ell*1e3:.1f} mm")
print(f"  Vacuum f₁ = c/(2ℓ) = {C_LIGHT/(2*params.ell)/1e9:.2f} GHz\n")
print(f"  {'ε_eff':>8}  {'f₁ (no Δℓ)':>12}  {'f₁ (with Δℓ)':>12}  {'Z₀':>8}")
print(f"  {'':>8}  {'[GHz]':>12}  {'[GHz]':>12}  {'[Ω]':>8}")
print("  " + "-" * 50)

for eps_test in [1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 10.0]:
    p_test = DesignParams(**{**params.__dict__, 'eps_eff': eps_test})
    m_no = eigenfrequencies(p_test, 1, eps_eff=eps_test, include_end_correction=False)
    m_ec = eigenfrequencies(p_test, 1, eps_eff=eps_test, include_end_correction=True)
    Z0_test = Z0_with_dielectric(Z0_flat_stripline(params.w, params.R), eps_test)
    print(f"  {eps_test:8.1f}  {m_no['f_GHz'][0]:10.3f}    {m_ec['f_GHz'][0]:10.3f}    {Z0_test:6.1f}")

print("\n  → From HFSS eigenmode: extract f₁, then ε_eff = [c/(2ℓ_eff × f₁)]²")
print("    This calibrates all analytic formulas for sweeps and higher modes.")




████████████████████████████████████████████████████████████████████████
  PART 2: FREQUENCY vs ε_eff (FOR HFSS CALIBRATION)
████████████████████████████████████████████████████████████████████████

  Strip length ℓ = 12.0 mm
  Vacuum f₁ = c/(2ℓ) = 12.49 GHz

     ε_eff    f₁ (no Δℓ)  f₁ (with Δℓ)        Z₀
                   [GHz]         [GHz]       [Ω]
  --------------------------------------------------
       1.0      12.491        11.813     204.4
       2.0       8.833         8.357     144.6
       3.0       7.212         6.824     118.0
       4.0       6.246         5.910     102.2
       5.0       5.586         5.286      91.4
       6.0       5.100         4.826      83.5
       7.0       4.721         4.468      77.3
       8.0       4.416         4.179      72.3
      10.0       3.950         3.738      64.7

  → From HFSS eigenmode: extract f₁, then ε_eff = [c/(2ℓ_eff × f₁)]²
    This calibrates all analytic formulas for sweeps and higher modes.


#### 3. WITH USER-SPECIFIED ε_eff (typical for this geometry)


In [11]:

# Based on similar Axline/Ganjam devices: ε_eff ~ 5 is typical for
# sapphire in a ~2.5 mm tunnel. Use this for the "best estimate."

print("\n" + "█" * 72)
print("  PART 3: BEST-ESTIMATE ANALYSIS (ε_eff = 5.0)")
print("█" * 72 + "\n")

params_best = DesignParams(
    R          = 1.25e-3,
    ell        = 12.0e-3,
    ell_cyl    = 34.0e-3,
    w          = 100e-6,
    t          = 200e-9,
    h_sub      = 430e-6,
    eps_r_sub  = 10.0,
    T          = 20e-3,
    T_c        = 1.2,
    R_s_residual = 5e-9,
    eps_eff    = 5.0,          # <-- representative override
)

results_best = full_analysis(
    params_best,
    n_modes=5,
    include_end_correction=True,
    tan_delta_bulk=1e-8,
    p_bulk=0.3,
    tan_delta_surf=2e-3,
    p_surf=1e-4,
    verbose=True,
)




████████████████████████████████████████████████████████████████████████
  PART 3: BEST-ESTIMATE ANALYSIS (ε_eff = 5.0)
████████████████████████████████████████████████████████████████████████

STRIPLINE-IN-CYLINDER RESONATOR: ANALYTIC ESTIMATES

⚠  VALIDITY WARNINGS:
   • CAUTION: w/(2R) = 0.0400 < 0.05. Pozar's effective-width correction is outside its fitted range. Using W_e ≈ w (no correction) is safer for very narrow strips. Z₀ estimate may be off by ~20%.

  Enclosure:  R = 1.250 mm  (d = 2.50 mm)
  Strip:      ℓ = 12.00 mm,  w = 100.0 µm
  Tunnel:     ℓ_cyl = 34.0 mm
  Substrate:  h = 430 µm,  ε_r = 10.0
  Gaps:       g₁ = 11.00 mm,  g₂ = 11.00 mm

--- Waveguide cutoff ---
  TE₁₁: 70.3 GHz
  TM₀₁: 91.8 GHz
  γ⁻¹ at f₁: 0.681 mm
  γ×g₁ = 16.2  →  e⁻²ᵞᵍ = 9.3e-15

--- Characteristic impedance ---
  Z₀ (vacuum, flat stripline): 204.4 Ω
  Z₀ (vacuum, coax equiv):     234.7 Ω
  ε_eff = 5.00  (user override)
  Z₀ (with ε_eff):             91.4 Ω

--- Eigenfrequencies (first 5 modes) 

#### 4. PARAMETER SWEEPS: design sensitivities


In [12]:


print("\n" + "█" * 72)
print("  PART 4: DESIGN SENSITIVITIES")
print("█" * 72 + "\n")

# 4a. Frequency vs strip length (primary tuning knob)
print("--- 4a. f₁ vs strip length (ε_eff = 5.0) ---\n")
print(f"  {'ℓ [mm]':>8}  {'f₁ [GHz]':>10}  {'Δℓ [mm]':>10}  {'f₁/f₁(12mm)':>12}")
print("  " + "-" * 48)

# Compute reference frequency at ℓ = 12 mm first
p_ref = DesignParams(**{**params_best.__dict__, 'ell': 12e-3})
m_ref = eigenfrequencies(p_ref, 1, eps_eff=5.0, include_end_correction=True)
f1_ref = m_ref['f_GHz'][0]

for ell_mm in [8, 9, 10, 11, 12, 13, 14, 15, 16]:
    p_sweep = DesignParams(**{**params_best.__dict__, 'ell': ell_mm * 1e-3})
    m = eigenfrequencies(p_sweep, 1, eps_eff=5.0, include_end_correction=True)
    ratio = m['f_GHz'][0] / f1_ref
    print(f"  {ell_mm:8d}  {m['f_GHz'][0]:8.3f}    "
          f"{(m['delta_ell_1']+m['delta_ell_2'])*1e3:8.3f}    {ratio:10.4f}")

print(f"\n  Scaling: f₁ ∝ 1/ℓ  (to ~3% from end correction)")

# 4b. Frequency vs enclosure diameter
print("\n--- 4b. f₁ vs enclosure diameter (ε_eff = 5.0, ℓ = 12 mm) ---\n")
print(f"  {'d [mm]':>8}  {'f₁ [GHz]':>10}  {'f_c [GHz]':>10}  {'γ⁻¹ [mm]':>10}  {'Δf/f [%]':>10}")
print("  " + "-" * 56)

for d_mm in [1.5, 2.0, 2.5, 3.0, 4.0, 5.0, 6.0, 8.0, 10.0, 15.0]:
    R_test = d_mm / 2 * 1e-3
    p_sweep = DesignParams(**{**params_best.__dict__, 'R': R_test})
    m = eigenfrequencies(p_sweep, 1, eps_eff=5.0, include_end_correction=True)
    m_no = eigenfrequencies(p_sweep, 1, eps_eff=5.0, include_end_correction=False)
    fc = cutoff_frequency(R_test)
    gamma_test = evanescent_gamma(R_test, m['f_Hz'][0])
    dl_pct = (m_no['f_GHz'][0] - m['f_GHz'][0]) / m_no['f_GHz'][0] * 100
    print(f"  {d_mm:8.1f}  {m['f_GHz'][0]:8.3f}    {fc/1e9:8.1f}    "
          f"{1e3/gamma_test if gamma_test > 0 else float('inf'):8.3f}    {dl_pct:8.2f}")

print(f"\n  Key: larger d → lower f_c, longer γ⁻¹, larger Δℓ, lower f₁")
print(f"  For d > ~8 mm, end correction becomes a significant fraction of ℓ.")
print(f"  For d > ~15 mm, harmonics may hit cutoff — TEM model breaks down.")


# 4c. Strip width effect on Z₀ and Q
print("\n--- 4c. Z₀ and Q_cond vs strip width (d = 2.5 mm, ε_eff = 5.0) ---\n")
print(f"  {'w [µm]':>8}  {'Z₀_vac [Ω]':>12}  {'Z₀_eff [Ω]':>12}  {'Q_cond':>12}  {'w/(2R)':>8}")
print("  " + "-" * 60)

for w_um in [10, 25, 50, 100, 200, 500, 1000]:
    w_test = w_um * 1e-6
    Z0v = Z0_flat_stripline(w_test, params_best.R)
    Z0e = Z0_with_dielectric(Z0v, 5.0)
    # Q at fundamental
    v_p = C_LIGHT / np.sqrt(5.0)
    beta_test = 2 * np.pi * 5.5e9 / v_p
    Q_test = Q_conductor(beta_test, Z0e, w_test, params_best.R, 5e-9)
    ratio_wR = w_test / (2 * params_best.R)
    flag = " ⚠" if ratio_wR > 0.3 else ""
    print(f"  {w_um:8d}  {Z0v:10.1f}    {Z0e:10.1f}    {Q_test:10.2e}    "
          f"{ratio_wR:6.3f}{flag}")

print(f"\n  ⚠ = w/(2R) > 0.3: flat-stripline approximation unreliable")




████████████████████████████████████████████████████████████████████████
  PART 4: DESIGN SENSITIVITIES
████████████████████████████████████████████████████████████████████████

--- 4a. f₁ vs strip length (ε_eff = 5.0) ---

    ℓ [mm]    f₁ [GHz]     Δℓ [mm]   f₁/f₁(12mm)
  ------------------------------------------------
         8     7.720       0.683        1.4604
         9     6.924       0.682        1.3097
        10     6.276       0.682        1.1872
        11     5.739       0.681        1.0856
        12     5.286       0.681        1.0000
        13     4.900       0.681        0.9269
        14     4.566       0.680        0.8638
        15     4.275       0.680        0.8087
        16     4.019       0.680        0.7602

  Scaling: f₁ ∝ 1/ℓ  (to ~3% from end correction)

--- 4b. f₁ vs enclosure diameter (ε_eff = 5.0, ℓ = 12 mm) ---

    d [mm]    f₁ [GHz]   f_c [GHz]    γ⁻¹ [mm]    Δf/f [%]
  --------------------------------------------------------
       1.5     5.40

#### 5. THREE-RESONATOR PLANNING


In [13]:


print("\n" + "█" * 72)
print("  PART 5: THREE-RESONATOR FREQUENCY PLANNING")
print("█" * 72 + "\n")

# Target: 3 resonators at different frequencies in a 34 mm tunnel
# Typical cQED frequencies: 4-7 GHz, well separated (> 0.5 GHz)

target_freqs = [4.5, 5.5, 6.5]

plan = plan_multi_resonator(target_freqs, params_best, eps_eff=5.0, min_gap_mm=3.0)

print(f"  Target frequencies: {plan['target_freqs_GHz']} GHz")
print(f"  Required strip lengths:")
for i, (f, l) in enumerate(zip(plan['target_freqs_GHz'], plan['strip_lengths_mm'])):
    print(f"    Resonator {i+1}: f = {f:.1f} GHz  →  ℓ ≈ {l:.2f} mm")
print(f"\n  Total strip:  {plan['total_strip_mm']:.1f} mm")
print(f"  Total gaps:   {plan['total_gaps_mm']:.1f} mm  "
      f"({plan['n_resonators']+1} gaps × 3 mm)")
print(f"  Total needed: {plan['total_needed_mm']:.1f} mm")
print(f"  Tunnel:       {plan['tunnel_length_mm']:.1f} mm")
print(f"  Margin:       {plan['margin_mm']:.1f} mm  "
      f"{'✓ FITS' if plan['fits'] else '✗ DOES NOT FIT'}")
print(f"\n  Inter-strip isolation: {plan['inter_strip_isolation_dB']:.0f} dB  "
      f"(at 3 mm gap, {max(target_freqs):.1f} GHz)")

# Also try tighter gaps
plan_tight = plan_multi_resonator(target_freqs, params_best, eps_eff=5.0, min_gap_mm=2.0)
print(f"\n  With 2 mm gaps:")
print(f"    Total needed: {plan_tight['total_needed_mm']:.1f} mm  "
      f"{'✓ FITS' if plan_tight['fits'] else '✗ DOES NOT FIT'}")
print(f"    Margin: {plan_tight['margin_mm']:.1f} mm")
print(f"    Isolation: {plan_tight['inter_strip_isolation_dB']:.0f} dB")

# Important finding: with ε_eff = 5, individual strips are too long.
print("\n  *** 3 straight strips at 4.5-6.5 GHz do NOT fit in 34 mm ***")
print("  with ε_eff = 5. Possible solutions:")
print("    1. Higher ε_eff (thicker/wider substrate → more filling)")
print("    2. Higher frequencies (shorter strips)")
print("    3. Folded/hairpin geometry (Ganjam) to reduce footprint")
print("    4. Longer tunnel")
print("    5. The actual ε_eff from HFSS may differ — recalculate once known")

print("\n  Retry with f = [5.5, 6.5, 7.5] GHz, ε_eff = 7.0, 2 mm gaps:")
plan_v2 = plan_multi_resonator([5.5, 6.5, 7.5], params_best, eps_eff=7.0, min_gap_mm=2.0)
for i, (f, l) in enumerate(zip(plan_v2['target_freqs_GHz'], plan_v2['strip_lengths_mm'])):
    print(f"    Res {i+1}: f = {f:.1f} GHz  →  ℓ ≈ {l:.2f} mm")
print(f"    Total: {plan_v2['total_needed_mm']:.1f} / "
      f"{plan_v2['tunnel_length_mm']:.1f} mm  "
      f"{'✓ FITS' if plan_v2['fits'] else '✗ NO FIT'}  "
      f"(margin: {plan_v2['margin_mm']:.1f} mm)")
print(f"    Isolation: {plan_v2['inter_strip_isolation_dB']:.0f} dB")


# %% =========================================================================
# 6. HFSS INTERPRETATION GUIDE
# =============================================================================

print("\n" + "█" * 72)
print("  PART 6: HFSS INTERPRETATION CHECKLIST")
print("█" * 72 + "\n")

f1_best = results_best['modes_with_dielectric']['f_GHz'][0]
Z0_best = results_best['Z0_with_eps_eff']

print(f"""
  When you run HFSS eigenmode on this geometry, compare:

  1. EIGENFREQUENCY
     Analytic f₁ = {f1_best:.3f} GHz  (ε_eff = 5.0 assumed)
     If HFSS gives f₁_HFSS, extract:
       ε_eff = [c / (2 × ℓ_eff × f₁_HFSS)]²
       where ℓ_eff = ℓ + 2Δℓ ≈ {results_best['modes_with_dielectric']['ell_eff']*1e3:.3f} mm
     Or simpler: ε_eff ≈ ({C_LIGHT/(2*params_best.ell)/1e9:.2f} / f₁_HFSS)²
     (ignoring end correction — it's a ~3% effect)

  2. CHARACTERISTIC IMPEDANCE (from 2D port solve)
     Analytic Z₀ = {Z0_best:.1f} Ω  (flat stripline + ε_eff = 5.0)
     HFSS 2D port solve gives the exact cross-sectional Z₀.
     Expect HFSS value to be ~10-30% different due to curvature.

  3. Q FACTOR
     Analytic Q_cond = {results_best['Q_budget']['Q_conductor']:.2e}
     This is an UPPER BOUND. Your measured Q will be limited by:
       Q_surf ~ {results_best['Q_budget']['Q_surface_dielectric']:.2e}  (surface TLS)
       Q_bulk ~ {results_best['Q_budget']['Q_bulk_dielectric']:.2e}  (bulk sapphire)
     Axline achieved Q_i up to ~10⁸ (undercoupled), ~10⁷ (with qubit).
     Ganjam achieved Q_i ~ 3×10⁷ in optimized devices.

  4. MODE FIELD PATTERN
     - Verify the λ/2 TEM pattern: E-field max at strip center,
       nodes at strip ends
     - Check for stray modes: there should be NO modes below f_c
       = {results_best['f_c_TE11_GHz']:.1f} GHz other than TEM harmonics
     - If you see unexpected modes, check for seams, chip edge
       effects, or coupling pin artifacts

  5. PARTICIPATION RATIOS (from field integration)
     p_bulk  = ∫_sapphire ε|E|² dV  /  ∫_all ε|E|² dV
     p_surf  = (t_surf/ε_surf) × ∮_interface |E|² dA  /  ∫_all ε|E|² dV
     These feed directly into the Q model:
       1/Q_i = Σ p_i × Γ_i
""")

print("=" * 72)
print("  END OF ANALYSIS")
print("=" * 72)


████████████████████████████████████████████████████████████████████████
  PART 5: THREE-RESONATOR FREQUENCY PLANNING
████████████████████████████████████████████████████████████████████████

  Target frequencies: [4.5, 5.5, 6.5] GHz
  Required strip lengths:
    Resonator 1: f = 4.5 GHz  →  ℓ ≈ 14.90 mm
    Resonator 2: f = 5.5 GHz  →  ℓ ≈ 12.19 mm
    Resonator 3: f = 6.5 GHz  →  ℓ ≈ 10.31 mm

  Total strip:  37.4 mm
  Total gaps:   12.0 mm  (4 gaps × 3 mm)
  Total needed: 49.4 mm
  Tunnel:       34.0 mm
  Margin:       -15.4 mm  ✗ DOES NOT FIT

  Inter-strip isolation: 76 dB  (at 3 mm gap, 6.5 GHz)

  With 2 mm gaps:
    Total needed: 45.4 mm  ✗ DOES NOT FIT
    Margin: -11.4 mm
    Isolation: 51 dB

  *** 3 straight strips at 4.5-6.5 GHz do NOT fit in 34 mm ***
  with ε_eff = 5. Possible solutions:
    1. Higher ε_eff (thicker/wider substrate → more filling)
    2. Higher frequencies (shorter strips)
    3. Folded/hairpin geometry (Ganjam) to reduce footprint
    4. Longer tunnel
